# Text2Event–mT5 adaptation — GENEVA
Flow: Kaggle import → Colab GPU → pretrained mT5 → 5 substructure + 1 full epoch on all TRAIN → fixed TRAIN32/DEV32 evaluation → save checkpoints and results to Drive.


In [ ]:
import os
from pathlib import Path
IS_COLAB = Path('/content').exists() and not os.environ.get('KAGGLE_KERNEL_RUN_TYPE')
if not IS_COLAB:
    print('Open in Colab to run.')
else:
    import sys, subprocess, json, hashlib, zipfile, shutil, datetime
    subprocess.run([sys.executable,'-m','pip','install','-q','kagglehub==1.0.3','transformers==5.14.1','sentencepiece==0.2.1'],check=True)
    import kagglehub
    kagglehub.login()
    DATASET_HANDLE='vietquan299/text2event-constraint-ablation-20261027/versions/75'
    dataset_root=Path(kagglehub.dataset_download(DATASET_HANDLE))
    def digest(path):
        h=hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
        return h.hexdigest()
    ROOT=Path('/content/geneva-short-runtime')
    if ROOT.exists():shutil.rmtree(ROOT)
    archives=[p for p in dataset_root.rglob('geneva-colab-short.zip') if digest(p)=='16baeef1a7c7c970bb95c76fb9e6e5d5c8f0a3633e2f9b8e167b5dab1e62f903']
    if len(archives)==1:
        ROOT.mkdir()
        with zipfile.ZipFile(archives[0]) as z:
            for name in z.namelist():assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
            z.extractall(ROOT)
    else:
        manifests=[p for p in dataset_root.rglob('manifest.json') if digest(p)=='1e8c008cab78b832124b15a3ab77e6829cafd513907219a5f13b3613ff2446df']
        assert len(manifests)==1,'Pinned runtime not found'
        shutil.copytree(manifests[0].parent,ROOT)
    assert digest(ROOT/'manifest.json')=='1e8c008cab78b832124b15a3ab77e6829cafd513907219a5f13b3613ff2446df'
    for name,h in json.loads((ROOT/'manifest.json').read_text()).items():assert digest(ROOT/name)==h, 'Runtime hash differs: '+name
    import torch
    assert torch.cuda.is_available(),'GPU runtime required'
    print('GPU_PREFLIGHT',torch.__version__,torch.cuda.device_count())
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ID='GENEVA-short-fresh-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    output=Path('/content/drive/MyDrive/Event-Extraction')/RUN_ID
    output.parent.mkdir(parents=True,exist_ok=True)
    assert not output.exists(),'Output already exists'
    command=[sys.executable,'-u',str(ROOT/'runner/run_text2event_short_vhe.py'),
        '--source',str(ROOT/'source'),'--lock',str(ROOT/'locks/vhe-runtime-lock.json'),
        '--dataset','GENEVA','--output',str(output),'--full-epochs','1','--eval-samples','32',
        '--sample-stall-seconds','120','--progress-stall-seconds','300','--io-stall-seconds','900']
    print('FRESH_RUN',RUN_ID,'5 substructure + 1 full; TRAIN32/DEV32',flush=True)
    launcher_log=output.parent/(RUN_ID+'-launcher.log')
    with launcher_log.open('w') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        returncode=process.wait()
    if returncode:
        status_path=output/'status.json'
        if status_path.exists():print(status_path.read_text(),flush=True)
        pilot_log=output/'pilot.log'
        if pilot_log.exists():print(pilot_log.read_text()[-12000:],flush=True)
        print('Launcher log:',str(launcher_log),flush=True)
        raise RuntimeError('Runner failed; see the traceback above. Saved logs retained.')
    status=json.loads((output/'status.json').read_text());print('STATUS',json.dumps(status,ensure_ascii=False))
    results_path=output/'results.json'
    if results_path.exists():
        results=json.loads(results_path.read_text())
        for stage_epoch,result in results.items():
            print(stage_epoch,'DEV32',json.dumps(result['dev'],ensure_ascii=False))
    print('Drive output:',str(output))
    if status['status']!='short_pilot_complete':
        raise RuntimeError('Run interrupted; inspect status.json and pilot.log.')
